# Use of Songs Added in the 1991 Edition

This notebook asks a historical question with data: after the 1991 Denson edition added new songs, which of those songs actually became part of regular singing practice? It looks at the question from several angles: yearly use, song-by-song trajectories, songs that were later removed, and regional differences.

The basic method is to treat the minutes database like a long record of choices. Every time a song appears as a lesson in the minutes, that is evidence that singers chose that song at that singing. By counting those choices over time, we can see whether a song was only tried briefly, kept a steady place, grew later, or became popular mainly in certain regions.

Metric notes:

- A **lesson** is counted as one distinct `(minutes_id, lesson_id, song_id)`, so two people jointly leading the same song in the same lesson do not double count the song.
- A **singing record/day** is counted as one `minutes` row. Some rows cover multi-day conventions, so treat this as a practical database unit rather than a perfect day-level measure.
- `minutes.Year` is the minute-book year, not always the singing year. The notebook derives `singing_year` from `DateOrdinal` when available and otherwise parses `minutes.Date`.
- Most charts use **share of lessons**, not just raw counts. That matters because the database has more records in some years than others; shares make years easier to compare.


In [ ]:
from pathlib import Path
import datetime as dt
import itertools
import os
import re
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import seaborn as sns
sns.set_theme(style="whitegrid")

GITHUB_REPO = os.environ.get("FASOLA_NOTEBOOK_REPO", "marktgodfrey/fasolaminutes_parsing")
GITHUB_REF = os.environ.get("FASOLA_NOTEBOOK_REF", "pre1995")
RESOURCE_CACHE = Path(os.environ.get("FASOLA_NOTEBOOK_RESOURCE_DIR", ".notebook_resources"))
REFRESH_NOTEBOOK_RESOURCES = os.environ.get("FASOLA_NOTEBOOK_REFRESH", "").lower() in {"1", "true", "yes"}


def github_resource(relative_path: str) -> Path:
    relative_path = relative_path.lstrip("/")
    target = RESOURCE_CACHE / relative_path
    if REFRESH_NOTEBOOK_RESOURCES or not target.exists():
        import urllib.parse
        import urllib.request

        target.parent.mkdir(parents=True, exist_ok=True)
        quoted_path = urllib.parse.quote(relative_path)
        url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}/{quoted_path}"
        tmp_target = target.with_suffix(target.suffix + ".download")
        urllib.request.urlretrieve(url, tmp_target)
        tmp_target.replace(target)
        print(f"Downloaded {relative_path} from GitHub.")
    return target

DB_PATH = github_resource("minutes_pre95.db")
NEW_SONGS_PATH = github_resource("1991newsongs.txt")
REMOVED_SONGS_PATH = github_resource("1991removedsongs.txt")

## Load Song And Minutes Data

This first data step gathers the raw ingredients. The song table tells us which page or song number each lesson refers to, and the minutes tables tell us when and where the lesson appeared.

In plain terms, this is like putting the index cards in order before doing a history project: one stack says "what song was sung," another says "which singing record it came from," and later sections join those stacks together so each lesson has both a song identity and a year.


In [ ]:
def read_1991_new_songs(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        page, title = line.split(" ", 1)
        rows.append({"page_num": page, "listed_title": title})
    return pd.DataFrame(rows)

conn = sqlite3.connect(DB_PATH)

new_song_pages = read_1991_new_songs(NEW_SONGS_PATH)

songs_1991 = pd.read_sql_query(
    """
    SELECT bsj.page_num, bsj.song_id, songs.title
    FROM book_song_joins bsj
    JOIN books ON books.id = bsj.book_id
    JOIN songs ON songs.id = bsj.song_id
    WHERE books.year = 1991
    """,
    conn,
)

new_songs = new_song_pages.merge(songs_1991, on="page_num", how="left")
missing = new_songs[new_songs.song_id.isna()]
assert missing.empty, missing
new_songs["song_id"] = new_songs["song_id"].astype(int)
new_song_ids = set(new_songs.song_id)

minutes = pd.read_sql_query(
    """
    SELECT id AS minutes_id, Name, Location, Date, Year AS book_year,
           DateOrdinal, DensonYear, IsDenson, IsVirtual
    FROM minutes
    WHERE IsDenson = 1 AND COALESCE(IsVirtual, 0) = 0
    """,
    conn,
)

lessons = pd.read_sql_query(
    """
    SELECT DISTINCT minutes_id, lesson_id, song_id
    FROM song_leader_joins
    WHERE minutes_id IS NOT NULL AND lesson_id IS NOT NULL AND song_id IS NOT NULL
    """,
    conn,
)

leaders_by_minutes = pd.read_sql_query(
    """
    SELECT DISTINCT minutes_id, leader_id
    FROM song_leader_joins
    WHERE minutes_id IS NOT NULL AND leader_id IS NOT NULL
    """,
    conn,
)

location_rows = pd.read_sql_query(
    """
    SELECT mlj.minutes_id,
           GROUP_CONCAT(DISTINCT locations.state_province) AS state_province,
           GROUP_CONCAT(DISTINCT locations.country) AS country
    FROM minutes_location_joins mlj
    JOIN locations ON locations.id = mlj.location_id
    GROUP BY mlj.minutes_id
    """,
    conn,
)

conn.close()

new_songs.head(), minutes.shape, lessons.shape

## Derive Singing Year

The year printed in the minutes data is useful, but it is not always the exact year of the singing event. This section creates a cleaner `singing_year` field by using `DateOrdinal` when possible, then falling back to the written date when needed.

This matters because the notebook compares song use across time. If a convention was entered under one bookkeeping year but actually happened near the edge of another year, using the best available event date keeps the timeline more accurate.


In [ ]:
MONTHS = "January February March April May June July August September October November December".split()
DATE_RE = re.compile(r"(" + "|".join(MONTHS) + r")\s+(\d{1,2}).*?(\d{4})", re.I)
MONTH_RE = re.compile(r"(" + "|".join(MONTHS) + r")", re.I)
CURRENT_YEAR = dt.date.today().year


def plausible_singing_year(year, book_year=None):
    if pd.isna(year):
        return False
    year = int(year)
    if year < 1980 or year > CURRENT_YEAR + 1:
        return False
    if pd.notna(book_year) and abs(year - int(book_year)) > 1:
        return False
    return True


def fallback_year_from_book(row):
    if pd.isna(row.book_year):
        return np.nan
    book_year = int(row.book_year)
    month_match = MONTH_RE.search(str(row.Date))
    if month_match and month_match.group(1).lower() == "december":
        return book_year - 1
    return book_year


def singing_year_from_row(row) -> float:
    ordinal_year = np.nan
    if pd.notna(row.DateOrdinal):
        try:
            ordinal_year = dt.date.fromordinal(int(row.DateOrdinal)).year
        except ValueError:
            ordinal_year = np.nan
    if plausible_singing_year(ordinal_year, row.book_year):
        return int(ordinal_year)

    match = DATE_RE.search(str(row.Date))
    text_year = int(match.group(3)) if match else np.nan
    if plausible_singing_year(text_year, row.book_year):
        return int(text_year)

    return fallback_year_from_book(row)

STATE_ALIASES = {
    "alabama": "AL", "al": "AL", "georgia": "GA", "ga": "GA",
    "mississippi": "MS", "ms": "MS", "tennessee": "TN", "tn": "TN",
    "florida": "FL", "fl": "FL", "texas": "TX", "tx": "TX",
    "louisiana": "LA", "la": "LA", "arkansas": "AR", "ar": "AR",
    "north carolina": "NC", "nc": "NC", "south carolina": "SC", "sc": "SC",
    "virginia": "VA", "va": "VA", "west virginia": "WV", "wv": "WV",
    "kentucky": "KY", "ky": "KY", "illinois": "IL", "il": "IL",
    "indiana": "IN", "in": "IN", "ohio": "OH", "oh": "OH",
    "michigan": "MI", "mi": "MI", "wisconsin": "WI", "wi": "WI",
    "minnesota": "MN", "mn": "MN", "missouri": "MO", "mo": "MO",
    "iowa": "IA", "ia": "IA", "kansas": "KS", "ks": "KS",
    "california": "CA", "ca": "CA", "oregon": "OR", "or": "OR",
    "washington": "WA", "wa": "WA", "colorado": "CO", "co": "CO",
    "new mexico": "NM", "nm": "NM", "arizona": "AZ", "az": "AZ",
    "new york": "NY", "ny": "NY", "massachusetts": "MA", "ma": "MA",
    "connecticut": "CT", "ct": "CT", "vermont": "VT", "vt": "VT",
    "maine": "ME", "me": "ME", "pennsylvania": "PA", "pa": "PA",
    "maryland": "MD", "md": "MD", "oregon": "OR", "ireland": "IE",
    "united kingdom": "UK", "england": "UK", "ontario": "ON", "canada": "CA-Canada",
}

def normalize_state(value):
    if pd.isna(value) or not str(value).strip():
        return None
    first = str(value).split(",")[0].strip()
    key = first.lower().replace(".", "")
    return STATE_ALIASES.get(key, first.upper() if len(first) == 2 else first)

def infer_state_from_text(*parts):
    text = " ".join(str(p) for p in parts if pd.notna(p)).lower().replace(".", " ")
    # Prefer long names before abbreviations so "Georgia" beats a stray "GA"-like token.
    for key in sorted(STATE_ALIASES, key=len, reverse=True):
        if re.search(r"\b" + re.escape(key) + r"\b", text):
            return STATE_ALIASES[key]
    return None

minutes = minutes.merge(location_rows, on="minutes_id", how="left")
minutes["singing_year"] = minutes.apply(singing_year_from_row, axis=1)
minutes["state"] = minutes["state_province"].map(normalize_state)
missing_state = minutes["state"].isna()
minutes.loc[missing_state, "state"] = minutes.loc[missing_state].apply(
    lambda row: infer_state_from_text(row.Location, row.Name), axis=1
)

year_audit = minutes.loc[
    (minutes.singing_year < 1980) |
    (minutes.singing_year > CURRENT_YEAR + 1) |
    (minutes.book_year.notna() & ((minutes.singing_year - minutes.book_year).abs() > 1)),
    ["minutes_id", "Name", "Date", "book_year", "DateOrdinal", "singing_year"],
]
assert year_audit.empty, year_audit

minutes[["minutes_id", "Name", "Date", "book_year", "singing_year", "state"]].head(10)

## Popularity And Change Among All Songs

Before focusing only on the 1991 additions, this section looks at the whole repertory from 1992 through 2025. That gives a background picture of what "popular," "stable," "rising," and "fading" look like across all songs, not just the new ones.

The method is straightforward: count lessons by song and year, convert those counts into yearly shares, then compare songs across the same time window. The yearly share is the percentage of all lessons in that year that used a given song. This keeps a big database year from automatically looking more important than a small database year.

This section asks three high-level questions:

- Which songs were used the most overall?
- Which songs stayed popular year after year instead of spiking once?
- Which songs changed the most between the early and late parts of the period?


In [ ]:
ANALYSIS_START_YEAR = 1992
ANALYSIS_END_YEAR = min(2025, int(minutes.singing_year.max()))
analysis_years = list(range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1))

lesson_facts = lessons.merge(
    minutes[["minutes_id", "singing_year"]],
    on="minutes_id",
    how="inner",
)
lesson_facts = lesson_facts.dropna(subset=["singing_year"])
lesson_facts["singing_year"] = lesson_facts["singing_year"].astype(int)
lesson_facts = lesson_facts[lesson_facts.singing_year.between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)].copy()
lesson_facts["is_1991_new"] = lesson_facts.song_id.isin(new_song_ids)

all_analysis_song_ids = sorted(lesson_facts.song_id.dropna().astype(int).unique())
all_song_lookup = pd.DataFrame({"song_id": all_analysis_song_ids}).merge(
    songs_1991[["song_id", "page_num", "title"]].drop_duplicates("song_id"),
    on="song_id",
    how="left",
)
all_song_lookup["page_num"] = all_song_lookup.page_num.fillna("").astype(str)
all_song_lookup["title"] = all_song_lookup.title.fillna("song_id " + all_song_lookup.song_id.astype(str))
all_song_lookup["song_label"] = np.where(
    all_song_lookup.page_num.ne(""),
    all_song_lookup.page_num + " " + all_song_lookup.title,
    all_song_lookup.title,
)

year_total = lesson_facts.groupby("singing_year").agg(
    total_lessons=("song_id", "size"),
    singing_records=("minutes_id", "nunique"),
)

song_year_counts = (
    lesson_facts
    .groupby(["song_id", "singing_year"])
    .agg(
        lessons=("song_id", "size"),
        song_records=("minutes_id", "nunique"),
    )
    .reset_index()
)

all_song_years = pd.MultiIndex.from_product(
    [all_analysis_song_ids, analysis_years],
    names=["song_id", "singing_year"],
).to_frame(index=False)

song_year_grid = (
    all_song_years
    .merge(song_year_counts, on=["song_id", "singing_year"], how="left")
    .merge(year_total.reset_index(), on="singing_year", how="left")
    .merge(all_song_lookup, on="song_id", how="left")
)
song_year_grid[["lessons", "song_records"]] = song_year_grid[["lessons", "song_records"]].fillna(0).astype(int)
song_year_grid["share_of_all_lessons"] = song_year_grid.lessons / song_year_grid.total_lessons
song_year_grid["annual_rank"] = song_year_grid.groupby("singing_year").lessons.rank(
    method="min",
    ascending=False,
)
song_year_grid.loc[song_year_grid.lessons.eq(0), "annual_rank"] = np.nan

all_top_songs_period = (
    song_year_grid.groupby(["song_id", "page_num", "title", "song_label"])
    .agg(
        period_lessons=("lessons", "sum"),
        years_sung=("lessons", lambda values: int((values > 0).sum())),
        avg_annual_share=("share_of_all_lessons", "mean"),
        median_annual_rank=("annual_rank", "median"),
        top_5_years=("annual_rank", lambda values: int((values <= 5).sum())),
        top_10_years=("annual_rank", lambda values: int((values <= 10).sum())),
        top_25_years=("annual_rank", lambda values: int((values <= 25).sum())),
    )
    .sort_values("period_lessons", ascending=False)
    .reset_index()
)
all_top_songs_period["avg_annual_share_pct"] = all_top_songs_period.avg_annual_share * 100
all_top_songs_period.head(25)[[
    "page_num",
    "title",
    "period_lessons",
    "years_sung",
    "avg_annual_share_pct",
    "median_annual_rank",
    "top_5_years",
    "top_10_years",
    "top_25_years",
]]


### Consistently Popular Songs

This table starts with the overall top 25 songs and then asks how often each one appeared near the top in individual years. A song that appears in many annual top-5, top-10, or top-25 lists was not just helped by one unusually strong year; it kept showing up again and again.

Think of this as the difference between a student who gets one very high quiz score and a student who scores well all semester. Both may have a good average, but the second pattern shows consistency.


In [ ]:
consistent_popular = all_top_songs_period.head(25).sort_values(
    ["top_5_years", "top_10_years", "top_25_years", "period_lessons"],
    ascending=[False, False, False, False],
)

consistent_popular[[
    "page_num",
    "title",
    "period_lessons",
    "years_sung",
    "median_annual_rank",
    "top_5_years",
    "top_10_years",
    "top_25_years",
]].head(15)


### Top 25 Song Heatmap

This heatmap turns the top-song table into a picture. Each row is a song, each column is a year, and the color shows that song's share of all lessons in that year. Darker cells mean the song made up a larger part of that year's singing.

Rows are ordered by total lessons over the 1992-2025 period. Reading across a row shows a song's timeline; reading down a column shows which songs were especially prominent in one year.


In [ ]:
top25_ids = all_top_songs_period.head(25).song_id
top25_heat = song_year_grid[song_year_grid.song_id.isin(top25_ids)].copy()
top25_pivot = top25_heat.pivot_table(
    index="song_label",
    columns="singing_year",
    values="share_of_all_lessons",
    fill_value=0,
)
top25_order = all_top_songs_period.head(25).song_label
top25_pivot = top25_pivot.reindex(top25_order)

fig, ax = plt.subplots(figsize=(14, max(7, len(top25_pivot) * 0.34)))
sns.heatmap(top25_pivot * 100, cmap="mako", ax=ax, cbar_kws={"label": "% of all lessons"})
ax.set_title("Top 25 Songs by Year, 1992-2025")
ax.set_xlabel("Singing year")
ax.set_ylabel("")
plt.show()


### Songs That Faded Or Rose

This section looks for long-term change, not short-term noise. The 34-year period is split into three blocks: 1992-2002, 2003-2014, and 2015-2025. The main comparison is between the first and last blocks.

For each song, the notebook calculates its average annual share in the early block and its average annual share in the late block. Songs with a much lower late share are treated as fading; songs with a much higher late share are treated as rising. The middle block is kept as context, because some songs change gradually rather than jumping straight from low to high or high to low.


In [ ]:
early_years = range(ANALYSIS_START_YEAR, ANALYSIS_START_YEAR + 11)
late_years = range(ANALYSIS_END_YEAR - 10, ANALYSIS_END_YEAR + 1)
middle_years = range(ANALYSIS_START_YEAR + 11, ANALYSIS_END_YEAR - 10)

def summarize_year_block(years, prefix):
    block = song_year_grid[song_year_grid.singing_year.isin(list(years))]
    summary = block.groupby("song_id").agg(
        **{
            f"{prefix}_lessons": ("lessons", "sum"),
            f"{prefix}_avg_share": ("share_of_all_lessons", "mean"),
            f"{prefix}_years_sung": ("lessons", lambda values: int((values > 0).sum())),
        }
    )
    summary[f"{prefix}_rank"] = summary[f"{prefix}_avg_share"].rank(method="min", ascending=False)
    return summary

song_change = (
    all_song_lookup[["song_id", "page_num", "title"]]
    .merge(summarize_year_block(early_years, "early"), on="song_id", how="left")
    .merge(summarize_year_block(middle_years, "middle"), on="song_id", how="left")
    .merge(summarize_year_block(late_years, "late"), on="song_id", how="left")
    .merge(all_top_songs_period[["song_id", "period_lessons", "years_sung"]], on="song_id", how="left")
)
for col in [
    "early_lessons", "middle_lessons", "late_lessons",
    "early_avg_share", "middle_avg_share", "late_avg_share",
    "early_years_sung", "middle_years_sung", "late_years_sung",
    "period_lessons", "years_sung",
]:
    song_change[col] = song_change[col].fillna(0)

song_change["early_avg_share_pct"] = song_change.early_avg_share * 100
song_change["late_avg_share_pct"] = song_change.late_avg_share * 100
song_change["change_pct_points"] = song_change.late_avg_share_pct - song_change.early_avg_share_pct
song_change["absolute_change_pct_points"] = song_change.change_pct_points.abs()
song_change["rank_change"] = song_change.early_rank - song_change.late_rank

change_cols = [
    "page_num",
    "title",
    "period_lessons",
    "early_lessons",
    "late_lessons",
    "early_avg_share_pct",
    "late_avg_share_pct",
    "change_pct_points",
    "early_rank",
    "late_rank",
    "rank_change",
]

song_change.sort_values("absolute_change_pct_points", ascending=False)[change_cols].head(15)


In [ ]:
faded_from_start = song_change[song_change.early_lessons > 0].sort_values("change_pct_points").head(12)
rose_later = song_change.sort_values("change_pct_points", ascending=False).head(12)

display(faded_from_start[change_cols])
display(rose_later[change_cols])


### Faded And Rising Song Heatmaps

These heatmaps show the faded and rising song lists from the previous section. The goal is to make the change visible instead of relying only on table numbers.

The vertical guide lines mark the early, middle, and late comparison periods. If a "faded" song really faded, its row should usually look darker on the left and lighter on the right. If a "rising" song really rose, the opposite pattern should appear.


In [ ]:
def plot_change_heatmap(change_subset, title):
    heatmap_ids = change_subset.song_id.tolist()
    heatmap_data = song_year_grid[song_year_grid.song_id.isin(heatmap_ids)].copy()
    heatmap_pivot = heatmap_data.pivot_table(
        index="song_label",
        columns="singing_year",
        values="share_of_all_lessons",
        fill_value=0,
    )
    label_order = (
        change_subset
        .merge(all_song_lookup[["song_id", "song_label"]], on="song_id", how="left")
        .song_label
    )
    heatmap_pivot = heatmap_pivot.reindex(label_order)

    fig, ax = plt.subplots(figsize=(14, max(4.5, len(heatmap_pivot) * 0.42)))
    sns.heatmap(
        heatmap_pivot * 100,
        cmap="mako",
        ax=ax,
        cbar_kws={"label": "% of all lessons"},
    )
    for boundary_year in [min(middle_years), min(late_years)]:
        boundary_position = list(heatmap_pivot.columns).index(boundary_year)
        ax.axvline(boundary_position, color="white", lw=1.5, alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel("Singing year")
    ax.set_ylabel("")
    plt.show()

plot_change_heatmap(faded_from_start, "Songs That Faded From Early Use")
plot_change_heatmap(rose_later, "Songs That Rose In Later Use")


## Overall Yearly Use Of 1991 New Songs

Now the notebook focuses on the songs that were newly added in the 1991 edition. Instead of asking about one song at a time, this section asks how much the entire new-song group was used each year.

For each year, the notebook counts lessons from 1991-added songs and compares them with all lessons in that year. This produces measures like "what percent of lessons came from the new additions?" and "what percent of singing records included at least one new addition?"

This is a useful first view because it shows the general adoption curve. If the new songs were tried heavily right after publication and then dropped, the graph should show an early burst. If they became part of the regular repertory, the graph should show continued use later on.


In [ ]:
lesson_facts = lessons.merge(
    minutes[["minutes_id", "singing_year"]],
    on="minutes_id",
    how="inner",
)
lesson_facts = lesson_facts.dropna(subset=["singing_year"])
lesson_facts["singing_year"] = lesson_facts["singing_year"].astype(int)
lesson_facts["is_1991_new"] = lesson_facts.song_id.isin(new_song_ids)

year_total = lesson_facts.groupby("singing_year").agg(
    total_lessons=("song_id", "size"),
    singing_records=("minutes_id", "nunique"),
)
year_new = lesson_facts[lesson_facts.is_1991_new].groupby("singing_year").agg(
    new_song_lessons=("song_id", "size"),
    records_with_new_song=("minutes_id", "nunique"),
)

yearly = year_total.join(year_new, how="left").fillna(0)
yearly["share_of_lessons"] = yearly.new_song_lessons / yearly.total_lessons
yearly["share_of_records"] = yearly.records_with_new_song / yearly.singing_records
yearly = yearly.reset_index()

EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(exist_ok=True)

use_of_added_songs_export = yearly[[
    "singing_year",
    "share_of_lessons",
    "share_of_records",
    "new_song_lessons",
    "records_with_new_song",
    "total_lessons",
    "singing_records",
]].copy()
use_of_added_songs_export["share_of_lessons_pct"] = use_of_added_songs_export.share_of_lessons * 100
use_of_added_songs_export["share_of_records_pct"] = use_of_added_songs_export.share_of_records * 100
use_of_added_songs_export.to_csv(EXPORT_DIR / "use_of_songs_added_1991_edition.csv", index=False)

yearly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly.singing_year, yearly.share_of_lessons * 100, marker="o", label="% of all lessons")
# ax.plot(yearly.singing_year, yearly.share_of_records * 100, marker="o", label="% of singing records with any 1991 new song")
ax.axvline(1992, color="0.3", lw=1, alpha=0.5)
ax.set_title("Use of Songs Added in the 1991 Edition")
ax.set_xlabel("Singing year")
ax.set_ylabel("Share (%)")
ax.legend()
ax.margins(x=0.01)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly.singing_year, yearly.total_lessons, marker="o", label="All lessons")
ax.plot(yearly.singing_year, yearly.new_song_lessons, marker="o", label="1991 new song lessons")
ax.axvline(1992, color="0.3", lw=1, alpha=0.5)
ax.set_title("Lesson Counts for Songs Added in the 1991 Edition")
ax.set_xlabel("Singing year")
ax.set_ylabel("Lessons")
ax.legend()
ax.margins(x=0.01)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly.singing_year, yearly.share_of_records * 100, marker="o", label="% of singing records with any 1991 new song")
ax.axvline(1992, color="0.3", lw=1, alpha=0.5)
ax.set_title("Use of Songs Added in the 1991 Edition")
ax.set_xlabel("Singing year")
ax.set_ylabel("Share (%)")
ax.legend()
ax.margins(x=0.01)
plt.show()

### Initial Burst vs Later Use

This section groups individual years into broader eras so the overall pattern is easier to read. Year-by-year charts can be noisy, especially when the number of records changes from year to year. Era averages smooth out some of that noise.

The question is simple: were the 1991 additions mostly an early experiment, or did they keep being used after the first wave of attention passed? The year ranges are adjustable if another periodization makes better historical sense.


In [ ]:
bins = [1990, 1994, 1999, 2009, 2019, 2100]
labels = ["1992-1994", "1995-1999", "2000-2009", "2010-2019", "2020+"]
lesson_facts["era"] = pd.cut(lesson_facts.singing_year, bins=bins, labels=labels)
era_summary = lesson_facts.groupby("era", observed=True).agg(
    total_lessons=("song_id", "size"),
    new_song_lessons=("is_1991_new", "sum"),
    singing_records=("minutes_id", "nunique"),
)
era_days = lesson_facts[lesson_facts.is_1991_new].groupby("era", observed=True).minutes_id.nunique()
era_summary["records_with_new_song"] = era_days
era_summary = era_summary.fillna(0)
era_summary["share_of_lessons"] = era_summary.new_song_lessons / era_summary.total_lessons
era_summary["share_of_records"] = era_summary.records_with_new_song / era_summary.singing_records
era_summary

## Song-By-Song Trajectories of 1991 New Songs

This section breaks the new-song group apart. A group total can hide very different stories: one new song might become a favorite, another might disappear, and another might grow slowly over many years.

The method is to calculate each added song's yearly lesson count and yearly share, then plot those values across time. These trajectories let us compare adoption patterns song by song instead of treating all 1991 additions as if they behaved the same way.


In [ ]:
song_year_counts = lesson_facts[lesson_facts.song_id.isin(new_song_ids)].groupby(["song_id", "singing_year"]).agg(
    lessons=("song_id", "size"),
    singing_records=("minutes_id", "nunique"),
).reset_index()
song_year_counts = song_year_counts.merge(year_total.reset_index(), on="singing_year", how="left")
song_year_counts = song_year_counts.merge(new_songs[["song_id", "page_num", "title"]], on="song_id", how="left")
song_year_counts["share_of_all_lessons"] = song_year_counts.lessons / song_year_counts.total_lessons
song_year_counts["share_of_records"] = song_year_counts.singing_records_x / song_year_counts.singing_records_y
song_year_counts = song_year_counts.rename(columns={"singing_records_x": "song_records", "singing_records_y": "all_records"})

top_new_songs = (song_year_counts.groupby(["song_id", "page_num", "title"]).lessons.sum()
                 .sort_values(ascending=False).head(40).reset_index())
top_new_songs

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for _, song in top_new_songs.head(10).iterrows():
    group = song_year_counts[song_year_counts.song_id == song.song_id]
    label = f"{song.page_num} {song.title}"
    ax.plot(group.singing_year, group.share_of_all_lessons * 100, marker="o", ms=3, label=label)
ax.set_title("Top 1991 New Songs: Share of All Lessons")
ax.set_xlabel("Singing year")
ax.set_ylabel("Share of all lessons (%)")
ax.legend(fontsize=8, ncol=2)
plt.show()

In [ ]:
heat = song_year_counts.merge(top_new_songs.head(40)[["song_id"]], on="song_id")
heat["song_label"] = heat.page_num + " " + heat.title
pivot = heat.pivot_table(index="song_label", columns="singing_year", values="share_of_all_lessons", fill_value=0)
pivot = pivot.loc[heat.groupby("song_label").lessons.sum().sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, max(6, len(pivot) * 0.32)))
if sns:
    sns.heatmap(pivot * 100, cmap="mako", ax=ax, cbar_kws={"label": "% of all lessons"})
else:
    im = ax.imshow(pivot * 100, aspect="auto")
    ax.set_yticks(range(len(pivot.index)), pivot.index)
    ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=90)
    fig.colorbar(im, ax=ax, label="% of all lessons")
ax.set_title("Song-by-Year Heatmap for the Most-Used 1991 Additions")
ax.set_xlabel("Singing year")
ax.set_ylabel("")
plt.show()

## Did Some Songs Start Fast While Others Rose Later?

This section compares early attention with later staying power. For each 1991-added song, the notebook measures how much it was used soon after the edition appeared and how much it was used later.

That comparison separates several possible stories. Some songs may have started fast and stayed strong. Others may have received early curiosity but faded. Still others may have started quietly and become more common only after singers had time to learn them.


In [ ]:
def era_total_for_song(start, end):
    mask = song_year_counts.singing_year.between(start, end)
    return song_year_counts[mask].groupby("song_id").lessons.sum()

early = era_total_for_song(1992, 1999)
middle = era_total_for_song(2000, 2010)
late = era_total_for_song(2011, int(lesson_facts.singing_year.max()))

rise = new_songs[["song_id", "page_num", "title"]].copy()
rise["early_lessons"] = rise.song_id.map(early).fillna(0).astype(int)
rise["middle_lessons"] = rise.song_id.map(middle).fillna(0).astype(int)
rise["late_lessons"] = rise.song_id.map(late).fillna(0).astype(int)
rise["late_minus_early"] = rise.late_lessons - rise.early_lessons
rise["early_share_of_song"] = rise.early_lessons / rise[["early_lessons", "middle_lessons", "late_lessons"]].sum(axis=1).replace(0, np.nan)
rise.sort_values("early_lessons", ascending=False).head(12)

In [ ]:
rise.sort_values("late_minus_early", ascending=False).head(12)

## Overall Yearly Use Of 1991 Songs Removed in 2025

Some songs added in 1991 were removed from the 2025 edition. This section asks what their usage looked like before that removal.

The method is the same as for all 1991 additions: count their lessons by year and compare those counts with overall yearly singing activity. The point is not to decide why a song was removed, but to describe whether the removed songs were rarely used, declining, regionally limited, or still active in the minutes data.


In [ ]:
removed_song_pages = read_1991_new_songs(REMOVED_SONGS_PATH)
removed_song_pages["page_num"] = removed_song_pages.page_num.str.rstrip("*")

removed_songs = removed_song_pages.merge(songs_1991, on="page_num", how="left")
missing_removed = removed_songs[removed_songs.song_id.isna()]
assert missing_removed.empty, missing_removed
removed_songs["song_id"] = removed_songs["song_id"].astype(int)
removed_song_ids = set(removed_songs.song_id)

lesson_facts["is_1991_removed_2025"] = lesson_facts.song_id.isin(removed_song_ids)

year_removed = lesson_facts[lesson_facts.is_1991_removed_2025].groupby("singing_year").agg(
    removed_song_lessons=("song_id", "size"),
    records_with_removed_song=("minutes_id", "nunique"),
)

yearly_removed = year_total.join(year_removed, how="left").fillna(0)
yearly_removed["share_of_lessons"] = yearly_removed.removed_song_lessons / yearly_removed.total_lessons
yearly_removed["share_of_records"] = yearly_removed.records_with_removed_song / yearly_removed.singing_records
yearly_removed = yearly_removed.reset_index()

lesson_count_comparison = yearly[["singing_year", "total_lessons", "new_song_lessons"]].merge(
    yearly_removed[["singing_year", "removed_song_lessons"]],
    on="singing_year",
    how="left",
)
lesson_count_comparison["removed_song_lessons"] = lesson_count_comparison.removed_song_lessons.fillna(0).astype(int)
lesson_count_comparison.to_csv(
    EXPORT_DIR / "lesson_counts_added_all_removed_1991.csv",
    index=False,
)

yearly_removed.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly_removed.singing_year, yearly_removed.share_of_lessons * 100, marker="o", label="% of all lessons")
ax.axvline(2025, color="0.3", lw=1, alpha=0.5)
ax.set_title("Overall Yearly Use Of 1991 Songs Removed in 2025")
ax.set_xlabel("Singing year")
ax.set_ylabel("Share (%)")
ax.legend()
ax.margins(x=0.01)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(lesson_count_comparison.singing_year, lesson_count_comparison.total_lessons, marker="o", label="All lessons")
ax.plot(lesson_count_comparison.singing_year, lesson_count_comparison.new_song_lessons, marker="o", label="1991 new song lessons")
ax.plot(lesson_count_comparison.singing_year, lesson_count_comparison.removed_song_lessons, marker="o", label="1991 removed-in-2025 song lessons")
ax.axvline(2025, color="0.3", lw=1, alpha=0.5)
ax.set_title("Lesson Counts for Songs Added, All Lessons, and Songs Removed")
ax.set_xlabel("Singing year")
ax.set_ylabel("Lessons")
ax.legend()
ax.margins(x=0.01)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly_removed.singing_year, yearly_removed.share_of_records * 100, marker="o", label="% of singing records with any removed-in-2025 song")
ax.axvline(2025, color="0.3", lw=1, alpha=0.5)
ax.set_title("Singing Records Using 1991 Songs Removed in 2025")
ax.set_xlabel("Singing year")
ax.set_ylabel("Share (%)")
ax.legend()
ax.margins(x=0.01)
plt.show()

## Song-By-Song Trajectories of 1991 Songs Removed From 2025

This section looks individually at the 1991-added songs that were later removed. The group-level chart can show the overall trend, but individual plots show whether all removed songs behaved similarly or whether the group contains different patterns.

Read these plots as timelines. A steady low line suggests a song never became common in the recorded minutes. A high early line that drops suggests fading use. A line with isolated spikes may mean the song was important in particular years or places rather than broadly common.


In [ ]:
removed_song_year_counts = lesson_facts[lesson_facts.song_id.isin(removed_song_ids)].groupby(["song_id", "singing_year"]).agg(
    lessons=("song_id", "size"),
    singing_records=("minutes_id", "nunique"),
).reset_index()
removed_song_year_counts = removed_song_year_counts.merge(year_total.reset_index(), on="singing_year", how="left")
removed_song_year_counts = removed_song_year_counts.merge(removed_songs[["song_id", "page_num", "title"]], on="song_id", how="left")
removed_song_year_counts["share_of_all_lessons"] = removed_song_year_counts.lessons / removed_song_year_counts.total_lessons
removed_song_year_counts["share_of_records"] = removed_song_year_counts.singing_records_x / removed_song_year_counts.singing_records_y
removed_song_year_counts = removed_song_year_counts.rename(columns={"singing_records_x": "song_records", "singing_records_y": "all_records"})

top_removed_songs = (removed_song_year_counts.groupby(["song_id", "page_num", "title"]).lessons.sum()
                     .sort_values(ascending=False).head(40).reset_index())
top_removed_songs

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for _, song in top_removed_songs.head(10).iterrows():
    group = removed_song_year_counts[removed_song_year_counts.song_id == song.song_id]
    label = f"{song.page_num} {song.title}"
    ax.plot(group.singing_year, group.share_of_all_lessons * 100, marker="o", ms=3, label=label)
ax.axvline(2025, color="0.3", lw=1, alpha=0.5)
ax.set_title("Top 1991 Songs Removed in 2025: Share of All Lessons")
ax.set_xlabel("Singing year")
ax.set_ylabel("Share of all lessons (%)")
ax.legend(fontsize=8, ncol=2)
plt.show()

In [ ]:
removed_heat = removed_song_year_counts.merge(top_removed_songs.head(40)[["song_id"]], on="song_id")
removed_heat["song_label"] = removed_heat.page_num + " " + removed_heat.title
removed_pivot = removed_heat.pivot_table(index="song_label", columns="singing_year", values="share_of_all_lessons", fill_value=0)
removed_pivot = removed_pivot.loc[removed_heat.groupby("song_label").lessons.sum().sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, max(6, len(removed_pivot) * 0.32)))
if sns:
    sns.heatmap(removed_pivot * 100, cmap="mako", ax=ax, cbar_kws={"label": "% of all lessons"})
else:
    im = ax.imshow(removed_pivot * 100, aspect="auto")
    ax.set_yticks(range(len(removed_pivot.index)), removed_pivot.index)
    ax.set_xticks(range(len(removed_pivot.columns)), removed_pivot.columns, rotation=90)
    fig.colorbar(im, ax=ax, label="% of all lessons")
ax.axvline(len(removed_pivot.columns) - 0.5, color="0.3", lw=1, alpha=0.5)
ax.set_title("Song-by-Year Heatmap for 1991 Songs Removed in 2025")
ax.set_xlabel("Singing year")
ax.set_ylabel("")
plt.show()

## Regional Popularity Across Co-Attendance Regions

This section asks whether the 1991 additions were used broadly across the singing world or mainly in particular co-attendance regions. A co-attendance region is a group of singings that share leaders often enough to look connected in the `singing_neighborhoods` analysis.

If `singing_neighborhood_regions.csv` exists, this section uses the canonical mapping in `Singings.csv` to attach each lesson to one of those regions. Then it compares a song's regional pattern with the background pattern of all lessons.

The key idea is fairness. Larger regions naturally have more lessons, so a song should not be called "regional" just because many of its lessons are in a large region. The balance score asks whether the song follows the normal regional distribution or is unusually concentrated somewhere.


In [ ]:
REGIONS_PATH = github_resource("singing_neighborhood_regions.csv")
REGION_NAMES_PATH = github_resource("singing_neighborhood_region_names.csv")
SINGINGS_MAP_PATH = github_resource("Singings.csv")

MIN_REGION_SONG_LESSONS = 8
MIN_ASSOCIATION_LESSONS_IN_REGION = 3
MIN_ASSOCIATION_LIFT = 2.0

if not REGIONS_PATH.exists():
    print(f"Skipping regional analysis: {REGIONS_PATH} does not exist.")
elif not SINGINGS_MAP_PATH.exists():
    print(f"Skipping regional analysis: {SINGINGS_MAP_PATH} does not exist.")
else:
    region_assignments = pd.read_csv(REGIONS_PATH)
    region_names = pd.read_csv(REGION_NAMES_PATH) if REGION_NAMES_PATH.exists() else None
    singing_map = pd.read_csv(SINGINGS_MAP_PATH)

    singing_map["canonical_singing"] = (
        singing_map["corrected_singing"]
        .fillna(singing_map["singing"])
        .astype(str)
        .str.strip()
    )
    singing_map = singing_map[
        singing_map["id"].notna()
        & singing_map["canonical_singing"].notna()
        & singing_map["canonical_singing"].ne("")
        & singing_map["canonical_singing"].ne("nan")
    ].copy()
    singing_map["minutes_id"] = singing_map["id"].astype(int)
    mapped_minutes = singing_map[["minutes_id", "canonical_singing"]].drop_duplicates()

    region_lookup = region_assignments[["singing", "region_id", "region_size", "mapped_minutes", "distinct_leaders"]].copy()
    minute_regions = mapped_minutes.merge(
        region_lookup,
        left_on="canonical_singing",
        right_on="singing",
        how="inner",
    )
    minute_regions = minute_regions[["minutes_id", "canonical_singing", "region_id", "region_size"]].drop_duplicates()

    fallback_region_labels = (
        region_assignments.sort_values(["region_id", "mapped_minutes"], ascending=[True, False])
        .groupby("region_id")
        .head(3)
        .groupby("region_id")["singing"]
        .apply(lambda values: "; ".join(values))
        .rename("region_label")
    )
    if region_names is not None:
        region_labels = (
            region_names.set_index("region_id")["region_name"]
            .rename("region_label")
            .combine_first(fallback_region_labels)
        )
    else:
        region_labels = fallback_region_labels

    regional_lesson_facts = lesson_facts.merge(minute_regions, on="minutes_id", how="inner")
    regional_lesson_facts = regional_lesson_facts.drop_duplicates(
        ["minutes_id", "lesson_id", "song_id", "region_id"]
    )

    mapped_minutes_count = regional_lesson_facts["minutes_id"].nunique()
    all_minutes_count = lesson_facts["minutes_id"].nunique()
    print(f"regional mapped minutes with lessons: {mapped_minutes_count:,} of {all_minutes_count:,} ({mapped_minutes_count / all_minutes_count:.1%})")
    print(f"regions represented: {regional_lesson_facts['region_id'].nunique():,}")

    region_totals = regional_lesson_facts.groupby("region_id").agg(
        region_total_lessons=("song_id", "size"),
        region_singing_records=("minutes_id", "nunique"),
    )
    region_totals = region_totals.join(region_labels, how="left")
    region_totals["background_lesson_share"] = region_totals.region_total_lessons / region_totals.region_total_lessons.sum()

    display(region_totals.sort_values("region_total_lessons", ascending=False).head(12))

### Balance Score

`regional_balance_score` measures how evenly a song's regional use matches the overall regional opportunity pattern. Technically, it is `1 - normalized Jensen-Shannon distance` between two distributions: the song's regional lesson distribution and the all-lesson regional background.

In simpler terms, imagine comparing two pie charts. One pie chart shows where all lessons came from. The other shows where this song's lessons came from. If the two pies have similar slices, the song has a high balance score. If one slice is much bigger for the song than expected, the score is lower.

Companion columns make the score easier to read:

- `top_region_lesson_share`: share of the song's mapped regional lessons in its strongest single region.
- `effective_regions`: entropy-based count of regions implied by the observed distribution.
- `max_lift`: strongest regional over-representation compared with the background expectation.


In [ ]:
def normalized_js_balance(observed_counts: pd.Series, background_probs: pd.Series) -> float:
    observed = observed_counts.reindex(background_probs.index, fill_value=0).astype(float)
    if observed.sum() <= 0:
        return np.nan
    p = observed / observed.sum()
    q = background_probs.astype(float) / background_probs.sum()
    m = 0.5 * (p + q)

    def kl(left, right):
        mask = left > 0
        return float((left[mask] * np.log(left[mask] / right[mask])).sum())

    js_divergence = 0.5 * kl(p, m) + 0.5 * kl(q, m)
    js_distance = np.sqrt(js_divergence)
    max_distance = np.sqrt(np.log(2))
    return 1 - (js_distance / max_distance)


def effective_region_count(counts: pd.Series) -> float:
    counts = counts[counts > 0].astype(float)
    if counts.empty:
        return np.nan
    probs = counts / counts.sum()
    return float(np.exp(-(probs * np.log(probs)).sum()))


if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    new_song_region_counts = (
        regional_lesson_facts[regional_lesson_facts.song_id.isin(new_song_ids)]
        .groupby(["song_id", "region_id"])
        .agg(
            song_region_lessons=("song_id", "size"),
            song_region_records=("minutes_id", "nunique"),
            song_region_singings=("canonical_singing", "nunique"),
        )
        .reset_index()
    )

    new_song_region_counts = new_song_region_counts.merge(region_totals.reset_index(), on="region_id", how="left")
    new_song_region_counts["regional_share_of_lessons"] = (
        new_song_region_counts.song_region_lessons / new_song_region_counts.region_total_lessons
    )

    total_regional_lessons = region_totals.region_total_lessons.sum()
    song_totals = new_song_region_counts.groupby("song_id").song_region_lessons.sum().rename("mapped_regional_lessons")
    new_song_region_counts = new_song_region_counts.merge(song_totals, on="song_id", how="left")
    new_song_region_counts["expected_lessons_if_balanced"] = (
        new_song_region_counts.mapped_regional_lessons
        * new_song_region_counts.region_total_lessons
        / total_regional_lessons
    )
    new_song_region_counts["lift_vs_regional_background"] = (
        new_song_region_counts.song_region_lessons / new_song_region_counts.expected_lessons_if_balanced
    )

    background_probs = region_totals["background_lesson_share"]
    balance_rows = []
    for song_id, group in new_song_region_counts.groupby("song_id"):
        counts = group.set_index("region_id")["song_region_lessons"]
        top_row = group.sort_values(
            ["song_region_lessons", "lift_vs_regional_background"],
            ascending=False,
        ).iloc[0]
        lift_regions = group[
            (group.song_region_lessons >= MIN_ASSOCIATION_LESSONS_IN_REGION)
            & (group.lift_vs_regional_background >= MIN_ASSOCIATION_LIFT)
        ].sort_values("lift_vs_regional_background", ascending=False)
        balance_rows.append({
            "song_id": song_id,
            "mapped_regional_lessons": int(counts.sum()),
            "regions_with_lessons": int((counts > 0).sum()),
            "effective_regions": effective_region_count(counts),
            "regional_balance_score": normalized_js_balance(counts, background_probs),
            "top_region_id": top_row.region_id,
            "top_region_label": top_row.region_label,
            "top_region_lessons": int(top_row.song_region_lessons),
            "top_region_lesson_share": top_row.song_region_lessons / counts.sum(),
            "max_lift": float(group.lift_vs_regional_background.max()),
            "associated_region_count": int(lift_regions.region_id.nunique()),
            "associated_regions": "; ".join(
                f"{row.region_id} ({row.lift_vs_regional_background:.1f}x, n={int(row.song_region_lessons)})"
                for row in lift_regions.itertuples()
            ),
        })

    song_region_balance = pd.DataFrame(balance_rows).merge(
        new_songs[["song_id", "page_num", "title"]],
        on="song_id",
        how="left",
    )
    song_region_balance = song_region_balance[
        song_region_balance.mapped_regional_lessons >= MIN_REGION_SONG_LESSONS
    ].sort_values("regional_balance_score", ascending=False)

    display_cols = [
        "page_num", "title",
        "mapped_regional_lessons", "regions_with_lessons",
        "effective_regions", "regional_balance_score",
        "top_region_id", "top_region_label", "top_region_lesson_share",
        "associated_region_count", "associated_regions",
        "max_lift"
    ]
    display(song_region_balance[display_cols].head(15))


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].hist(song_region_balance.regional_balance_score.dropna(), bins=20, color="steelblue", alpha=0.85)
    axes[0].set_title("Regional Balance Scores for 1991 Additions")
    axes[0].set_xlabel("balance score (1 = follows regional background)")
    axes[0].set_ylabel("songs")

    axes[1].scatter(
        song_region_balance.mapped_regional_lessons,
        song_region_balance.regional_balance_score,
        s=np.clip(song_region_balance.regions_with_lessons * 9, 20, 140),
        alpha=0.75,
    )
    axes[1].set_xscale("log")
    axes[1].set_title("Balance vs Mapped Regional Use")
    axes[1].set_xlabel("mapped regional lessons, log scale")
    axes[1].set_ylabel("regional balance score")

    plt.tight_layout()
    plt.show()


## Global Acceptance Of 1991 Additions

These views treat the 1991 additions as a song set rather than as individual songs. The main question is whether the new repertory became broadly accepted across co-attendance regions or whether adoption stayed uneven.

The notebook looks at acceptance in three related ways: whether regions used any 1991 additions, how much of each region's singing came from those additions, and how many different added songs each region used. Together, those measures distinguish "we tried one or two" from "these songs became a meaningful part of the local repertory."


### Regional Penetration Over Time

This section summarizes adoption by era and region. A region counts as penetrated in an era if it has mapped lessons from any 1991-added song. The percentage measures show how large a role those additions played within each region's total singing.

The heatmap is meant to be read like a classroom attendance chart: each row is a region, each column is an era, and stronger color means more use of the 1991 additions.


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    # acceptance_bins = [1990, 1994, 1999, 2009, 2019, 2100]
    # acceptance_labels = ["1992-1994", "1995-1999", "2000-2009", "2010-2019", "2020+"]

    acceptance_bins = [1990, 1994, 1999, 2004, 2009, 2014, 2019, 2100]
    acceptance_labels = ["1992-1994", "1995-1999", "2000-2004", "2005-2009", "2010-2014", "2015-2019", "2020+"]

    regional_acceptance_facts = regional_lesson_facts.dropna(subset=["singing_year"]).copy()
    regional_acceptance_facts["singing_year"] = regional_acceptance_facts["singing_year"].astype(int)
    regional_acceptance_facts["era"] = pd.cut(
        regional_acceptance_facts.singing_year,
        bins=acceptance_bins,
        labels=acceptance_labels,
    )
    regional_acceptance_facts["is_1991_new"] = regional_acceptance_facts.song_id.isin(new_song_ids)

    region_era_acceptance = regional_acceptance_facts.groupby(["region_id", "era"], observed=True).agg(
        total_lessons=("song_id", "size"),
        new_song_lessons=("is_1991_new", "sum"),
        singing_records=("minutes_id", "nunique"),
        new_songs_used=("song_id", lambda values: values[values.isin(new_song_ids)].nunique()),
    ).reset_index()
    region_era_acceptance["new_song_lesson_share"] = (
        region_era_acceptance.new_song_lessons / region_era_acceptance.total_lessons
    )
    region_era_acceptance["used_any_new_song"] = region_era_acceptance.new_song_lessons.gt(0)

    global_acceptance_era_summary = region_era_acceptance.groupby("era", observed=True).agg(
        regions_with_minutes=("region_id", "nunique"),
        regions_using_new_songs=("used_any_new_song", "sum"),
        total_lessons=("total_lessons", "sum"),
        new_song_lessons=("new_song_lessons", "sum"),
        median_region_new_song_share=("new_song_lesson_share", "median"),
        mean_region_new_song_share=("new_song_lesson_share", "mean"),
        median_new_songs_used=("new_songs_used", "median"),
    )
    global_acceptance_era_summary["pct_regions_using_new_songs"] = (
        global_acceptance_era_summary.regions_using_new_songs
        / global_acceptance_era_summary.regions_with_minutes
    )
    global_acceptance_era_summary["overall_new_song_lesson_share"] = (
        global_acceptance_era_summary.new_song_lessons
        / global_acceptance_era_summary.total_lessons
    )

    display(global_acceptance_era_summary[
        [
            "regions_with_minutes", "regions_using_new_songs", "pct_regions_using_new_songs",
            "overall_new_song_lesson_share", "median_region_new_song_share",
            "mean_region_new_song_share", "median_new_songs_used",
        ]
    ].style.format({
        "pct_regions_using_new_songs": "{:.1%}",
        "overall_new_song_lesson_share": "{:.2%}",
        "median_region_new_song_share": "{:.2%}",
        "mean_region_new_song_share": "{:.2%}",
        "median_new_songs_used": "{:.0f}",
    }))


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    heatmap_regions = (
        region_totals.sort_values("region_total_lessons", ascending=False)
        .head(40)
        .index
    )
    heatmap_data = region_era_acceptance[
        region_era_acceptance.region_id.isin(heatmap_regions)
    ].copy()
    heatmap_data = heatmap_data.merge(region_labels.reset_index(), on="region_id", how="left")
    heatmap_data["region_display"] = heatmap_data.region_id + " - " + heatmap_data.region_label.str.slice(0, 65)

    heatmap_pivot = heatmap_data.pivot_table(
        index="region_display",
        columns="era",
        values="new_song_lesson_share",
        fill_value=0,
        observed=True,
    )
    ordered_region_ids = region_totals.loc[heatmap_regions].sort_values("region_total_lessons", ascending=False).index
    region_display_order = (
        heatmap_data[["region_id", "region_display"]]
        .drop_duplicates()
        .set_index("region_id")
        .loc[ordered_region_ids, "region_display"]
    )
    heatmap_pivot = heatmap_pivot.reindex(region_display_order).iloc[:-7]

    fig, ax = plt.subplots(figsize=(11, max(8, len(heatmap_pivot) * 0.28)))
    sns.heatmap(
        heatmap_pivot * 100,
        cmap="viridis",
        ax=ax,
        cbar_kws={"label": "1991 additions as % of lessons"},
    )
    ax.set_xticklabels([label.get_text().replace("-", "-\n") for label in ax.get_xticklabels()], rotation=0)
    ax.set_title("Regional Adoption of 1991 Additions by Era")
    ax.set_xlabel("Era")
    ax.set_ylabel("Co-attendance region")
    plt.tight_layout()
    plt.show()


### Acceptance Distribution By Region

Each point in this view is one co-attendance region. The point shows how much that region used the 1991 additions, usually as a share of its total mapped lessons.

This helps answer whether acceptance was broad or uneven. If most regions cluster near the same value, adoption was fairly similar. If a few regions sit far above the rest, the overall total may be driven by those heavy-adopting regions.


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    region_acceptance_summary = regional_acceptance_facts.groupby("region_id").agg(
        total_lessons=("song_id", "size"),
        new_song_lessons=("is_1991_new", "sum"),
        singing_records=("minutes_id", "nunique"),
        new_songs_used=("song_id", lambda values: values[values.isin(new_song_ids)].nunique()),
    )
    region_acceptance_summary["new_song_lesson_share"] = (
        region_acceptance_summary.new_song_lessons / region_acceptance_summary.total_lessons
    )
    region_acceptance_summary["new_songs_used_pct"] = region_acceptance_summary.new_songs_used / len(new_song_ids)
    region_acceptance_summary = region_acceptance_summary.join(region_labels, how="left")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    sns.histplot(region_acceptance_summary.new_song_lesson_share * 100, bins=24, ax=axes[0], color="steelblue")
    axes[0].set_title("Regional Distribution of 1991-Addition Use")
    axes[0].set_xlabel("1991 additions as % of lessons")
    axes[0].set_ylabel("regions")

    plot_df = region_acceptance_summary[region_acceptance_summary.total_lessons >= 1000].reset_index()
    axes[1].scatter(
        plot_df.total_lessons,
        plot_df.new_song_lesson_share * 100,
        s=np.clip(plot_df.new_songs_used * 4, 18, 180),
        alpha=0.7,
    )
    axes[1].set_xscale("log")
    axes[1].set_title("Acceptance vs Region Size")
    axes[1].set_xlabel("total mapped lessons, log scale")
    axes[1].set_ylabel("1991 additions as % of lessons")

    plt.tight_layout()
    plt.show()

    display(region_acceptance_summary.sort_values(
        ["new_song_lesson_share", "total_lessons"],
        ascending=False,
    )[
        ["region_label", "total_lessons", "new_song_lessons", "new_song_lesson_share", "new_songs_used", "new_songs_used_pct"]
    ].head(15).style.format({
        "new_song_lesson_share": "{:.2%}",
        "new_songs_used_pct": "{:.1%}",
    }))


### Song-Set Coverage By Region

This section asks whether regions used many different 1991 additions or mostly leaned on a few favorites. Raw counts alone can be misleading, because a region might technically use ten songs while one song accounts for nearly all the lessons.

`effective_new_song_count` is an entropy-based measure. In plain language, it estimates how many songs were used in a meaningfully balanced way. It is high when lessons are spread across many added songs and lower when a small number of songs dominate.


In [ ]:
def effective_count(counts: pd.Series) -> float:
    counts = counts[counts > 0].astype(float)
    if counts.empty:
        return 0.0
    probs = counts / counts.sum()
    return float(np.exp(-(probs * np.log(probs)).sum()))


if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    region_song_set_counts = (
        regional_acceptance_facts[regional_acceptance_facts.is_1991_new]
        .groupby(["region_id", "song_id"])
        .size()
        .rename("new_song_lessons")
        .reset_index()
    )
    effective_new_song_counts = region_song_set_counts.groupby("region_id").new_song_lessons.apply(effective_count)

    region_new_song_coverage = region_acceptance_summary.copy()
    region_new_song_coverage["effective_new_song_count"] = effective_new_song_counts
    region_new_song_coverage["effective_new_song_count"] = region_new_song_coverage.effective_new_song_count.fillna(0)
    region_new_song_coverage["effective_new_song_pct"] = region_new_song_coverage.effective_new_song_count / len(new_song_ids)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    sns.histplot(region_new_song_coverage.new_songs_used, bins=20, ax=axes[0], color="darkseagreen")
    axes[0].set_title("How Many 1991 Additions Did Regions Use?")
    axes[0].set_xlabel("distinct 1991 additions used")
    axes[0].set_ylabel("regions")

    axes[1].scatter(
        region_new_song_coverage.new_songs_used,
        region_new_song_coverage.effective_new_song_count,
        s=np.clip(region_new_song_coverage.new_song_lessons / 2, 18, 180),
        alpha=0.7,
    )
    axes[1].plot([0, len(new_song_ids)], [0, len(new_song_ids)], color="0.4", lw=1, alpha=0.5)
    axes[1].set_title("Breadth vs Concentration Within the New-Song Set")
    axes[1].set_xlabel("distinct 1991 additions used")
    axes[1].set_ylabel("effective 1991 additions used")

    plt.tight_layout()
    plt.show()

    display(region_new_song_coverage.sort_values(
        ["new_songs_used", "effective_new_song_count", "new_song_lessons"],
        ascending=False,
    )[
        [
            "region_label", "new_song_lessons", "new_songs_used", "new_songs_used_pct",
            "effective_new_song_count", "effective_new_song_pct", "new_song_lesson_share",
        ]
    ].head(15).style.format({
        "new_songs_used_pct": "{:.1%}",
        "effective_new_song_count": "{:.1f}",
        "effective_new_song_pct": "{:.1%}",
        "new_song_lesson_share": "{:.2%}",
    }))


### Broadly Balanced vs Regionally Concentrated Songs

These tables compare two kinds of regional patterns among songs with enough mapped lessons to judge. The first table surfaces songs whose regional use looks most like the overall background distribution. The second surfaces songs whose use is most concentrated in particular regions.

The minimum lesson filter matters because a song with only a tiny number of mapped lessons can look extremely regional by accident. Requiring at least `MIN_REGION_SONG_LESSONS` lessons makes the comparison more trustworthy.


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    balanced_display = song_region_balance.sort_values(
        ["regional_balance_score", "mapped_regional_lessons"],
        ascending=[False, False],
    )[display_cols]
    display(balanced_display.head(12))


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    concentrated_display = song_region_balance.sort_values(
        # ["regions_with_lessons", "top_region_lesson_share", "max_lift", "mapped_regional_lessons"],
        # ascending=[True, False, False, False],
        ["regional_balance_score", "mapped_regional_lessons"],
        ascending=[True, False],
    )[display_cols + ["top_region_label"]]
    display(concentrated_display.head(15))


### Top-Region Share

This table uses a simpler concentration measure: for each song, what share of its mapped regional lessons are in its single strongest region?

This is easier to understand than the balance score, but less complete. It does not fully adjust for region size or the rest of the song's distribution. Use it as a quick "how concentrated is this?" check alongside the more careful balance score.


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    top_region_share_display = (
        song_region_balance.sort_values(
            ["top_region_lesson_share", "mapped_regional_lessons"],
            ascending=[False, False],
        )
        .assign(
            top_region_lesson_pct=lambda df: (df.top_region_lesson_share * 100).round(1),
            regional_balance_score=lambda df: df.regional_balance_score.round(3),
            max_lift=lambda df: df.max_lift.round(2),
        )
    )

    display(top_region_share_display[
        [
            "page_num", "title",
            "top_region_id", "top_region_label", "top_region_lessons", "top_region_lesson_pct",
            "regional_balance_score", "max_lift",
            "mapped_regional_lessons", "regions_with_lessons",
        ]
    ].head(20))


### Songs Unique To One Region

This section looks for the strictest possible regional concentration: songs whose mapped regional lessons all fall in one region. The lesson minimum prevents one stray data point from making a song look uniquely regional.

A song appearing here should be read as "unique within the mapped regional data used by this notebook," not necessarily unique in every real-world singing context. Missing mappings, sparse data, or unmapped records could change the picture.


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    unique_to_one_region = song_region_balance[
        song_region_balance.regions_with_lessons.eq(1)
    ].sort_values(["mapped_regional_lessons", "max_lift"], ascending=False)

    unique_display_cols = ["page_num", "title", "mapped_regional_lessons", "top_region_id", "top_region_label", "max_lift"]
    if unique_to_one_region.empty:
        print(
            "No 1991 additions are unique to a single mapped co-attendance region "
            f"at the current cutoff of {MIN_REGION_SONG_LESSONS} mapped regional lessons."
        )
        print("Showing the most regionally concentrated songs instead.")
        display(concentrated_display.head(15))
    else:
        display(unique_to_one_region[unique_display_cols].head(20))


### Songs Strongly Associated With Several Regions

This section looks for songs that are strongly over-represented in more than one region. A region counts as an association when the song has enough lessons there and appears at least `MIN_ASSOCIATION_LIFT` times more often than the background expectation.

These songs are not simply global favorites, but they are also not confined to one place. They may point to clusters of regional preference, travel patterns, or communities that share repertory habits.


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    multi_region_associations = song_region_balance[
        song_region_balance.associated_region_count.ge(2)
    ].sort_values(
        ["associated_region_count", "max_lift", "mapped_regional_lessons"],
        ascending=False,
    )

    display(multi_region_associations[display_cols].head(20))


In [ ]:
if REGIONS_PATH.exists() and SINGINGS_MAP_PATH.exists():
    interesting_song_ids = pd.concat([
        unique_to_one_region.head(5)[["song_id"]],
        multi_region_associations.head(5)[["song_id"]],
        song_region_balance.sort_values("regional_balance_score", ascending=False).head(5)[["song_id"]],
        song_region_balance.sort_values("regional_balance_score", ascending=True).head(5)[["song_id"]],
    ]).drop_duplicates().song_id.tolist()

    interesting_region_profiles = new_song_region_counts[
        new_song_region_counts.song_id.isin(interesting_song_ids)
    ].merge(
        new_songs[["song_id", "page_num", "title"]],
        on="song_id",
        how="left",
    ).sort_values(
        ["song_id", "song_region_lessons"],
        ascending=[True, False],
    )

    display(interesting_region_profiles[
        [
            "page_num", "title", "region_id", "region_label", "song_region_lessons",
            "regional_share_of_lessons", "lift_vs_regional_background", "song_region_records",
            "song_region_singings",
        ]
    ].head(80))
